In [8]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import CITY_COUNTRY, WEATHER_FILES, weather_path

#### Step 1: Load and stack all 11 weather files
**Setup:** imports pandas and our shared settings from `src/config.py`.

`weather_path("Lahore")` gives `Dataset/Pakistan/Weather/Lahore - Weather.csv`

In [9]:
missing = [c for c in WEATHER_FILES if not (ROOT / weather_path(c)).exists()]
print("Missing files:", missing if missing else "none, all 11 found")

for country in ["Pakistan", "India"]:
    folder = ROOT / "Dataset" / country / "Weather"
    print(country, sorted(p.name for p in folder.iterdir()))

Missing files: none, all 11 found
Pakistan ['Faisalabad - Weather.csv', 'Islamabad - Weather.csv', 'Lahore - Weather.csv', 'Multan - Weather.csv', 'Rahim Yar Khan - Weather.csv', 'Rawalpindi weather.csv', 'Sailkot-Weather.csv']
India ['Amritsar - Weather.csv', 'Ferozepur - Weather.csv', 'Jalandhar - Weather.csv', 'Ludhiana - Weather.xlsx']


**Check files exist:** makes sure all 11 files are where the config says, so a wrong name fails here.

In [10]:
RENAME = {
    "temperature_2m (°C)":      "temp_c",
    "relative_humidity_2m (%)": "rh_pct",
    "precipitation (mm)":       "precip_mm",
    "wind_speed_10m (km/h)":    "wind_speed_kmh",
    "wind_direction_10m (°)":   "wind_dir_deg",
}

def load_weather_file(city):
    path = ROOT / weather_path(city)
    reader = pd.read_excel if path.suffix == ".xlsx" else pd.read_csv

    meta = reader(path, nrows=1)        # header block: latitude, longitude, timezone
    df = reader(path, skiprows=3)       # skip the header block, real columns start after

    df = df.rename(columns=RENAME)      # rename BY NAME, so Ludhiana's column order doesn't matter
    assert set(RENAME.values()) <= set(df.columns), f"Missing columns for {city}"

    df["time"] = pd.to_datetime(df["time"])
    df.insert(0, "country", CITY_COUNTRY[city])
    df.insert(0, "city", city)

    centroid = {"city": city, "lat": meta["latitude"].iloc[0], "lon": meta["longitude"].iloc[0]}
    return df, centroid

**Loader function.** This does not load anything yet. It defines a reusable recipe, `load_weather_file(city)`, that we run once per city in the next cell.

What the recipe does for one city, in order:
1. Finds that city's file and opens it (CSV or Excel).
2. Skips the first 3 rows of the file. They hold the city's coordinates and timezone, not weather data, so we read them separately and keep only the latitude and longitude.
3. Renames the long column names to short ones.
4. Adds two new columns, `city` and `country`, so we still know whose data each row is after we stack the cities together.

Example for Lahore:
- Column name before: `temperature_2m (°C)`. After: `temp_c`.
- Two new columns added to every row: `city = Lahore`, `country = PK`.
- Coordinates saved on the side: `lat = 31.529`, `lon = 74.390`.

In [11]:
frames, centroids = [], []
for city in WEATHER_FILES:
    df, c = load_weather_file(city)
    frames.append(df)
    centroids.append(c)
    print(f"{city:15s} {len(df):7d} rows")

weather_raw = pd.concat(frames, ignore_index=True)
city_centroids = pd.DataFrame(centroids)

print("\nTotal shape:", weather_raw.shape)
display(weather_raw.head())
display(city_centroids)

Lahore            40896 rows
Faisalabad        40896 rows
Multan            40896 rows
Rawalpindi        40896 rows
Sialkot           40896 rows
Rahim Yar Khan    40896 rows
Islamabad         40896 rows
Amritsar          40512 rows
Ferozepur         40512 rows
Ludhiana          25920 rows
Jalandhar         40512 rows

Total shape: (433728, 8)


,city,country,time,temp_c,wind_speed_kmh,wind_dir_deg,precip_mm,rh_pct
0,Lahore,PK,2022-01-12 00:00:00,6.0,7.6,301,0.0,99
1,Lahore,PK,2022-01-12 01:00:00,6.5,5.8,300,0.0,98
2,Lahore,PK,2022-01-12 02:00:00,6.1,7.4,309,0.0,97
3,Lahore,PK,2022-01-12 03:00:00,7.9,5.9,322,0.0,92
4,Lahore,PK,2022-01-12 04:00:00,10.2,4.2,329,0.0,81


,city,lat,lon
0,Lahore,31.528997,74.389950
1,Faisalabad,31.458698,73.118280
2,Multan,30.123022,71.495330
3,Rawalpindi,33.567660,73.048325
4,Sialkot,32.513180,74.562040
5,Rahim Yar Khan,28.435852,70.261360
6,Islamabad,33.637960,73.027300
7,Amritsar,31.669594,74.892080
8,Ferozepur,30.966606,74.644550
9,Ludhiana,30.896300,75.834300


**Load all 11 and stack.** This cell runs the recipe from Cell 3 once for each of the 11 cities. Each run gives back one small table for that city. Then we join all 11 tables, one under the other, into one big table called `weather_raw`.

It also builds `city_centroids`, a small table with each city's latitude and longitude. Person C will need these later to measure how far each fire is from each city.

Example of the stacking:
- Before: 11 separate tables, for example Lahore has 40,896 rows and Ludhiana has 25,920 rows.
- After: one table, `weather_raw`, with all of them combined (433,728 rows). The `city` column tells us which rows belong to which city.

### Step 1 summary
- Loaded all 11 weather files (10 CSV and 1 Excel) with one loader function.
- Gave every row a `city` and `country` so the cities can be stacked into one table.
- Result: `weather_raw` with 433,728 rows and 8 columns, plus `city_centroids` with each city's coordinates.

## Step 2: Check each file

In [12]:
value_cols = ["temp_c", "rh_pct", "precip_mm", "wind_speed_kmh", "wind_dir_deg"]

**Column list.** Saves the names of the 5 weather columns in one list, so we can reuse it in the checks below. It prints nothing.

In [13]:
rows = []
for city, d in weather_raw.groupby("city"):
    first = d["time"].min()
    last = d["time"].max()
    rows.append({
        "city": city,
        "rows": len(d),
        "first_time": first,
        "last_time": last,
        "expected_rows": int((last - first) / pd.Timedelta(hours=1)) + 1,
        "duplicate_times": d["time"].duplicated().sum(),
        "empty_values": int(d[value_cols].isna().sum().sum()),
    })

file_check = pd.DataFrame(rows)
file_check

,city,rows,first_time,last_time,expected_rows,duplicate_times,empty_values
0,Amritsar,40512,2022-01-30,2026-09-13 23:00:00,40512,0,0
1,Faisalabad,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
2,Ferozepur,40512,2022-01-30,2026-09-13 23:00:00,40512,0,0
3,Islamabad,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
4,Jalandhar,40512,2022-01-30,2026-09-13 23:00:00,40512,0,0
5,Lahore,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
6,Ludhiana,25920,2022-01-15,2024-12-29 23:00:00,25920,0,0
7,Multan,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
8,Rahim Yar Khan,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
9,Rawalpindi,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0


**Per-city check.** Goes through the stacked table one city at a time and builds a small summary table, one row per city. For each city it records:
- how many rows it has, and its first and last timestamp
- `expected_rows`: how many hourly rows there should be between the first and last time
- how many timestamps appear twice, and how many empty values there are

If `rows` equals `expected_rows`, no hours are skipped.

In [14]:
full_frames = []
for city, d in weather_raw.groupby("city"):
    d = d.set_index("time")
    all_hours = pd.date_range(d.index.min(), d.index.max(), freq="h")
    d = d.reindex(all_hours)          # missing hours appear as empty rows
    d["city"] = city
    d["country"] = CITY_COUNTRY[city]
    d.index.name = "time"
    full_frames.append(d.reset_index())

weather_full = pd.concat(full_frames, ignore_index=True)
weather_full = weather_full[weather_raw.columns]   # keep the same column order as before

print("Before:", weather_raw.shape)
print("After: ", weather_full.shape)

Before: (433728, 8)
After:  (433728, 8)


**Full hourly calendar.** For each city, this makes a complete hour-by-hour list from its first to its last time, then lines the city's data up against that list. If an hour was skipped in the file, it now appears as an empty row (NaN) instead of silently disappearing. Then it stacks the 11 cities back together as `weather_full`.

Example: if a city had rows for 00:00, 01:00 and 03:00, it now has 00:00, 01:00, 02:00 (empty) and 03:00.

If the shape before and after is the same, nothing was skipped.

In [16]:
weather_full[value_cols].describe().round(1)

,temp_c,rh_pct,precip_mm,wind_speed_kmh,wind_dir_deg
count,433728.0,433728.0,433728.0,433728.0,433728.0
mean,24.0,62.0,0.1,7.4,181.0
std,8.8,23.1,0.6,4.1,108.6
min,0.5,2.0,0.0,0.0,0.0
25%,17.3,45.0,0.0,4.5,87.0
50%,25.4,64.0,0.0,6.8,170.0
75%,30.5,81.0,0.0,9.4,289.0
max,49.9,100.0,27.8,42.7,360.0


**Value range check.** Shows the count, average, minimum and maximum of each weather column, so we can spot impossible values.

What to look for: `rh_pct` should stay between 0 and 100, and `wind_dir_deg` between 0 and 360. Rain and wind speed should never be negative.

### Step 2 summary
- Checked each city: no duplicate timestamps, no skipped hours, no empty values.
- Built a full hourly calendar per city. Nothing was added, so the data had no gaps.
- Checked value ranges: humidity 2 to 100, wind direction 0 to 360, and no negative rain or wind.
- Result: `weather_full` is a complete and clean hourly table (433,728 rows).